# Tier-1 Autoencoder Agents (Multi-Modality)
This notebook implements the Unsupervised LSTM Autoencoder strategy for Tier-1, following a strict machine learning experimental methodology.

**Core Architecture Details:**
- **4 Independent Autoencoders** (Logon, Email, File, HTTP).
- **Deterministic Train/Val/Test Split:** 
  - Train: First 800 users (Normal behavior only)
  - Validation: Next 100 users (Used for Hyperparameter & Threshold Tuning)
  - Test: Final 100 users (Used ONLY for final evaluation)
- **MLflow Tracking:** Logs hyperparameters, metrics, and artifacts (ROC, Confusion Matrix).
- **Ablation Study:** Grid search over `hidden_dim` (16, 32, 64) and `sequence_length` (7, 14).
- **Regularization:** Dropout, Weight Decay, and Early Stopping.
- **Pure Anomaly Scores:** Exports reconstruction scores without handcrafted manual modifications.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay
import mlflow
import itertools
import gc
from pathlib import Path
import os
import matplotlib.pyplot as plt

# Dynamic Path Resolution

# === Determinism: seed all RNGs for reproducible results (as in week3/tier1_score_generation.ipynb) ===
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Use a seeded generator for DataLoader shuffle
g = torch.Generator()
g.manual_seed(SEED)

print(f"All RNGs seeded with SEED={SEED}")

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "pyproject.toml").exists() and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent

DATA_FILE = ROOT_DIR / "data" / "dayr4.2.csv"
MLFLOW_DB = f"sqlite:///{ROOT_DIR / 'mlflow.db'}"

mlflow.set_tracking_uri(MLFLOW_DB)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Global Deterministic Split

In [ ]:
print("Establishing Global User Split...")
df_users = pd.read_csv(DATA_FILE, usecols=['user'])
all_unique_users = sorted(df_users['user'].unique())

# Train: 800, Val: 100, Test: 100
TRAIN_USERS = set(all_unique_users[:800])
VAL_USERS = set(all_unique_users[800:900])
TEST_USERS = set(all_unique_users[900:1000])

print(f"Total Users: {len(all_unique_users)}")
print(f"Train Users: {len(TRAIN_USERS)}")
print(f"Val Users: {len(VAL_USERS)}")
print(f"Test Users: {len(TEST_USERS)}")
del df_users
gc.collect()

## 2. Dataset & Model

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, df, feature_cols, scaler, seq_len=14):
        self.features = scaler.transform(df[feature_cols].fillna(0).astype(np.float32))
        self.labels = (df['insider'].values > 0).astype(int)
        
        # Save raw data for exporting later
        self.users = df['user'].values
        self.weeks = df['week'].values
        self.days = df['day'].values
        
        self.seq_len = seq_len
        self.indices = []
        
        user_counts = df['user'].value_counts(sort=False)
        current_idx = 0
        for count in user_counts:
            if count >= seq_len:
                for i in range(count - seq_len + 1):
                    self.indices.append(current_idx + i)
            current_idx += count

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start_idx = self.indices[idx]
        seq = self.features[start_idx : start_idx + self.seq_len]
        label = self.labels[start_idx : start_idx + self.seq_len].max()
        
        # We need to return the user, week, day of the LAST day in the sequence for exporting
        end_idx = start_idx + self.seq_len - 1
        user_id = self.users[end_idx]
        week = self.weeks[end_idx]
        day = self.days[end_idx]
        
        return torch.tensor(seq), torch.tensor(seq), torch.tensor(label, dtype=torch.float32), user_id, week, day

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.2):
        super(LSTMAutoencoder, self).__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.decoder = nn.LSTM(hidden_dim, input_dim, batch_first=True)
        
    def forward(self, x):
        _, (h, _) = self.encoder(x)
        h = self.dropout(h)
        seq_len = x.shape[1]
        h_repeated = h[-1].unsqueeze(1).repeat(1, seq_len, 1)
        out, _ = self.decoder(h_repeated)
        return out

## 3. Hyperparameter Tuning & Export Loop

In [ ]:
modalities = ['logon', 'email', 'file_device', 'http']

param_grid = {
    'hidden_dim': [16, 32, 64],
    'sequence_length': [7, 14]
}

fixed_params = {
    'learning_rate': 0.001,
    'dropout': 0.2,
    'weight_decay': 1e-5,
    'epochs': 20, # Increased max epochs because of early stopping
    'optimizer': 'Adam'
}

all_test_predictions = []

def get_optimal_threshold(mses, labels):
    best_f1 = 0
    best_thresh = 0
    # Search percentiles from 90 to 99.9
    for p in np.linspace(90, 99.9, 100):
        thresh = np.percentile(mses, p)
        preds = (mses > thresh).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    
    # Fallback to 95th if F1 is 0 for all
    if best_thresh == 0:
        best_thresh = np.percentile(mses, 95)
        
    return best_thresh

for modality in modalities:
    print(f"\n{'='*50}")
    print(f"STARTING ABLATION STUDY FOR {modality.upper()}")
    print(f"{'='*50}")
    
    mlflow.set_experiment(f"Autoencoder_{modality.capitalize()}")
    
    with open(DATA_FILE, 'r') as f:
        all_cols = f.readline().strip().split(',')
        
    feature_cols = [c for c in all_cols if modality in c or c in ['O', 'C', 'E', 'A', 'N', 'role']]
    usecols = ['user', 'day', 'week', 'insider'] + feature_cols
    
    print(f"Loading {len(feature_cols)} features for {modality}...")
    df = pd.read_csv(DATA_FILE, usecols=usecols)
    df = df.sort_values(by=['user', 'day'])
    
    train_df = df[df['user'].isin(TRAIN_USERS)]
    val_df = df[df['user'].isin(VAL_USERS)]
    test_df = df[df['user'].isin(TEST_USERS)]
    
    train_normal = train_df[train_df['insider'] == 0]
    
    scaler = StandardScaler()
    scaler.fit(train_normal[feature_cols].fillna(0).astype(np.float32))
    
    best_val_f1 = -1
    best_val_auc = -1
    best_model_state = None
    best_params = None
    best_threshold = 0
    best_seq_len = 14
    
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    for run_idx, params in enumerate(combinations):
        seq_len = params['sequence_length']
        hidden_dim = params['hidden_dim']
        
        print(f"\n--- Run {run_idx+1}/{len(combinations)} | Seq: {seq_len}, Hidden: {hidden_dim} ---")
        
        with mlflow.start_run():
            mlflow.log_params(params)
            mlflow.log_params(fixed_params)
            
            # Create datasets
            train_dataset = SequenceDataset(train_normal, feature_cols, scaler, seq_len=seq_len)
            train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, generator=g)
            
            val_dataset = SequenceDataset(val_df, feature_cols, scaler, seq_len=seq_len)
            val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)
            
            model = LSTMAutoencoder(input_dim=len(feature_cols), hidden_dim=hidden_dim, dropout=fixed_params['dropout']).to(device)
            optimizer = optim.Adam(model.parameters(), lr=fixed_params['learning_rate'], weight_decay=fixed_params['weight_decay'])
            criterion = nn.MSELoss()
            
            # Training with Early Stopping
            best_train_loss = float('inf')
            patience = 3
            patience_counter = 0
            
            for epoch in range(fixed_params['epochs']):
                model.train()
                total_loss = 0
                for batch_x, batch_y, _, _, _, _ in train_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    optimizer.zero_grad()
                    out = model(batch_x)
                    loss = criterion(out, batch_y)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
                
                avg_loss = total_loss / len(train_loader)
                
                if avg_loss < best_train_loss:
                    best_train_loss = avg_loss
                    patience_counter = 0
                else:
                    patience_counter += 1
                    
                if patience_counter >= patience:
                    print(f"Early stopping triggered at epoch {epoch+1}")
                    break
            
            mlflow.log_metric("train_loss", best_train_loss)
            
            # Validation
            model.eval()
            val_mses, val_labels = [], []
            with torch.no_grad():
                for batch_x, _, labels, _, _, _ in val_loader:
                    batch_x = batch_x.to(device)
                    out = model(batch_x)
                    mse = torch.mean((out - batch_x)**2, dim=[1, 2]).cpu().numpy()
                    val_mses.extend(mse)
                    val_labels.extend(labels.numpy())
            
            # Threshold Tuning
            threshold = get_optimal_threshold(val_mses, val_labels)
            val_preds = (val_mses > threshold).astype(int)
            
            val_auc = roc_auc_score(val_labels, val_mses)
            val_f1 = f1_score(val_labels, val_preds, zero_division=0)
            
            mlflow.log_metric("val_auc", val_auc)
            mlflow.log_metric("val_f1", val_f1)
            mlflow.log_metric("threshold", threshold)
            
            print(f"Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f} | Thresh: {threshold:.4f}")
            
            if val_f1 > best_val_f1 or (val_f1 == best_val_f1 and val_auc > best_val_auc):
                best_val_f1 = val_f1
                best_val_auc = val_auc
                best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
                best_params = params
                best_threshold = threshold
                best_seq_len = seq_len
                
    # --- EVALUATE BEST MODEL ON TEST SET ---
    print(f"\n>> Best Params for {modality.upper()}: {best_params} | Val F1: {best_val_f1:.4f}")
    
    # Load best model
    best_model = LSTMAutoencoder(input_dim=len(feature_cols), hidden_dim=best_params['hidden_dim'], dropout=fixed_params['dropout']).to(device)
    best_model.load_state_dict(best_model_state)
    best_model.eval()
    
    test_dataset = SequenceDataset(test_df, feature_cols, scaler, seq_len=best_seq_len)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)
    
    test_mses, test_labels = [], []
    test_users, test_weeks, test_days = [], [], []
    
    with torch.no_grad():
        for batch_x, _, labels, u, w, d in test_loader:
            batch_x = batch_x.to(device)
            out = best_model(batch_x)
            mse = torch.mean((out - batch_x)**2, dim=[1, 2]).cpu().numpy()
            test_mses.extend(mse)
            test_labels.extend(labels.numpy())
            test_users.extend(u)
            test_weeks.extend(w.numpy())
            test_days.extend(d.numpy())
            
    # Calculate Final Test Metrics
    test_preds = (np.array(test_mses) > best_threshold).astype(int)
    test_auc = roc_auc_score(test_labels, test_mses)
    test_precision = precision_score(test_labels, test_preds, zero_division=0)
    test_recall = recall_score(test_labels, test_preds, zero_division=0)
    test_f1 = f1_score(test_labels, test_preds, zero_division=0)
    
    print(f"TEST METRICS -> AUC: {test_auc:.4f} | F1: {test_f1:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f}")
    
    # Generate Artifacts
    cm = confusion_matrix(test_labels, test_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    cm_path = ROOT_DIR / f"{modality}_confusion_matrix.png"
    plt.savefig(cm_path)
    plt.close()
    
    fpr, tpr, _ = roc_curve(test_labels, test_mses)
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {test_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic - {modality.upper()}')
    plt.legend(loc="lower right")
    roc_path = ROOT_DIR / f"{modality}_roc_curve.png"
    plt.savefig(roc_path)
    plt.close()
    
    # Also log to MLflow under the best run
    with mlflow.start_run(run_name=f"{modality}_best_test"):
        mlflow.log_params(best_params)
        mlflow.log_params(fixed_params)
        mlflow.log_metric("test_auc", test_auc)
        mlflow.log_metric("test_f1", test_f1)
        mlflow.log_metric("test_precision", test_precision)
        mlflow.log_metric("test_recall", test_recall)
        mlflow.log_artifact(cm_path)
        mlflow.log_artifact(roc_path)
        mlflow.set_tag("is_best_model", "true")
        
    os.remove(cm_path)
    os.remove(roc_path)
    
    # Store predictions for joining
    def normalize_to_prob(mse_scores, threshold, steepness=10):
        return 1 / (1 + np.exp(-(mse_scores - threshold) * steepness))
        
    test_probs = normalize_to_prob(np.array(test_mses), best_threshold)
    
    df_preds = pd.DataFrame({
        'user': test_users,
        'week': test_weeks,
        'day': test_days,
        'label': test_labels,
        f'{modality}_score': test_probs
    })
    
    # Deduplicate in case overlapping windows hit the same day
    df_preds = df_preds.groupby(['user', 'week', 'day', 'label'], as_index=False).max()
    all_test_predictions.append(df_preds)
    
    del df, train_df, val_df, test_df, train_normal
    gc.collect()

# 4. Merge and Save Final Tier 1 Scores
print("\nMerging final scores...")
dfs_indexed = [df.set_index(['user', 'week', 'day', 'label']) for df in all_test_predictions]
tier1_scores = pd.concat(dfs_indexed, axis=1).reset_index()
tier1_scores.fillna(0, inplace=True)
out_path = ROOT_DIR / "week3" / "tier1_scores.csv"
tier1_scores.to_csv(out_path, index=False)
print(f"Saved tier1_scores.csv to {out_path}")


In [ ]:
```
==================================================
STARTING ABLATION STUDY FOR LOGON
==================================================
Loading 21 features for logon...

--- Run 1/6 | Seq: 7, Hidden: 16 ---
Val AUC: 0.4956 | Val F1: 0.0422 | Thresh: 6.6535

--- Run 2/6 | Seq: 14, Hidden: 16 ---
Val AUC: 0.4869 | Val F1: 0.0411 | Thresh: 29.8113

--- Run 3/6 | Seq: 7, Hidden: 32 ---
Val AUC: 0.4918 | Val F1: 0.0422 | Thresh: 6.6446

--- Run 4/6 | Seq: 14, Hidden: 32 ---
Val AUC: 0.4781 | Val F1: 0.0411 | Thresh: 29.7201

--- Run 5/6 | Seq: 7, Hidden: 64 ---
Val AUC: 0.4913 | Val F1: 0.0422 | Thresh: 6.6372

--- Run 6/6 | Seq: 14, Hidden: 64 ---
Val AUC: 0.4785 | Val F1: 0.0411 | Thresh: 29.6941

>> Best Params for LOGON: {'hidden_dim': 16, 'sequence_length': 7} | Val F1: 0.0422
TEST METRICS -> AUC: 0.4833 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000

==================================================
STARTING ABLATION STUDY FOR EMAIL
==================================================
Loading 48 features for email...

--- Run 1/6 | Seq: 7, Hidden: 16 ---
Val AUC: 0.5733 | Val F1: 0.0131 | Thresh: 1.1192

--- Run 2/6 | Seq: 14, Hidden: 16 ---
Val AUC: 0.5572 | Val F1: 0.0091 | Thresh: 1.1184

--- Run 3/6 | Seq: 7, Hidden: 32 ---
Val AUC: 0.5763 | Val F1: 0.0123 | Thresh: 0.9968

--- Run 4/6 | Seq: 14, Hidden: 32 ---
Val AUC: 0.5664 | Val F1: 0.0100 | Thresh: 1.0000

--- Run 5/6 | Seq: 7, Hidden: 64 ---
Val AUC: 0.5792 | Val F1: 0.0130 | Thresh: 0.9699

--- Run 6/6 | Seq: 14, Hidden: 64 ---
Val AUC: 0.5667 | Val F1: 0.0100 | Thresh: 0.9529

>> Best Params for EMAIL: {'hidden_dim': 16, 'sequence_length': 7} | Val F1: 0.0131
TEST METRICS -> AUC: 0.4525 | F1: 0.0017 | Precision: 0.0009 | Recall: 0.0196

==================================================
STARTING ABLATION STUDY FOR FILE_DEVICE
==================================================
Loading 6 features for file_device...

--- Run 1/6 | Seq: 7, Hidden: 16 ---
Val AUC: 0.5098 | Val F1: 0.0706 | Thresh: 0.3571

--- Run 2/6 | Seq: 14, Hidden: 16 ---
Val AUC: 0.4980 | Val F1: 0.0713 | Thresh: 0.3364

--- Run 3/6 | Seq: 7, Hidden: 32 ---
Val AUC: 0.5007 | Val F1: 0.0706 | Thresh: 0.3437

--- Run 4/6 | Seq: 14, Hidden: 32 ---
Val AUC: 0.4965 | Val F1: 0.0713 | Thresh: 0.3253

--- Run 5/6 | Seq: 7, Hidden: 64 ---
Val AUC: 0.5074 | Val F1: 0.0706 | Thresh: 0.3399

--- Run 6/6 | Seq: 14, Hidden: 64 ---
Val AUC: 0.4968 | Val F1: 0.0713 | Thresh: 0.3253

>> Best Params for FILE_DEVICE: {'hidden_dim': 16, 'sequence_length': 14} | Val F1: 0.0713
TEST METRICS -> AUC: 0.4391 | F1: 0.0000 | Precision: 0.0000 | Recall: 0.0000

==================================================
STARTING ABLATION STUDY FOR HTTP
==================================================
Loading 195 features for http...

--- Run 1/6 | Seq: 7, Hidden: 16 ---
Val AUC: 0.6723 | Val F1: 0.1200 | Thresh: 4.3274

--- Run 2/6 | Seq: 14, Hidden: 16 ---
Val AUC: 0.6963 | Val F1: 0.1613 | Thresh: 3.4896

--- Run 3/6 | Seq: 7, Hidden: 32 ---
Val AUC: 0.6763 | Val F1: 0.1200 | Thresh: 4.2249

--- Run 4/6 | Seq: 14, Hidden: 32 ---
Val AUC: 0.6928 | Val F1: 0.1613 | Thresh: 3.4619

--- Run 5/6 | Seq: 7, Hidden: 64 ---
Val AUC: 0.6716 | Val F1: 0.1200 | Thresh: 4.1849

--- Run 6/6 | Seq: 14, Hidden: 64 ---
Val AUC: 0.6917 | Val F1: 0.1613 | Thresh: 3.3839

>> Best Params for HTTP: {'hidden_dim': 16, 'sequence_length': 14} | Val F1: 0.1613
TEST METRICS -> AUC: 0.6110 | F1: 0.1609 | Precision: 0.1109 | Recall: 0.2931

